Hard-negative / boundary undersampling (classical geometry baselines)

NearMiss (variants), Tomek links, Edited Nearest Neighbors (ENN) or Neighborhood Cleaning Rule (NCL).
These are standard “keep boundary points / clean majority” methods.

In [ ]:
# Load data (if not already loaded)
# Uncomment if needed:
import sys
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("DataLoad").getOrCreate()
df_2018 = spark.read.format("parquet").load("0917_2017_18_with_2017_cost.parquet")
df_og = df_2018.toPandas()

import importlib
import model_pipeline
importlib.reload(model_pipeline)
import pandas as pd
import numpy as np
import os, pickle
import model_IAI
importlib.reload(model_IAI)
from model_IAI import evaluate_binary_oct, finetune_oct


BIN_FLAG_COLUMNS = model_pipeline.get_bin_flag_columns(df_og) +['lab_monitoring_adherent','nephrology_consult_adherent','early_nephrology_referral']
STAGE_COLUMNS = [col for col in df_og.columns if "stage" in col.lower()]
#["stage_2017",'2017Q1_max_ckd_stage','2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage','2017Q4_max_ckd_stage']
CAT_COLUMNS = df_og.select_dtypes(include=["object","category"]).columns.tolist()
TRUE_NUM_COLUMNS = model_pipeline.get_true_num_columns(df_og,CAT_COLUMNS)+[ 'util_2017', 'total_increasing_quarters_2017'
, 'total_lab_tests', 'ckd_visit_count', 'quarters_with_labs', 'nephrology_visit_count', 'days_to_nephrology','MEDIAN_INCOME']
COST_COLUMNS = [col for col in df_og.columns if 
            "cost" in col.lower() or 
             "quarterly" in col.lower()  or "increasing" in col.lower()
             ]
UTILIZATION_COLUMNS = [col for col in df_og.columns if "claims" not in col.lower() ] + ['util_2017']
print("categorical cols: ", CAT_COLUMNS)
print("stage cols: ", STAGE_COLUMNS)
print(COST_COLUMNS)
leftover_cols = [
    c for c in df_og.columns 
    if c not in CAT_COLUMNS and c not in TRUE_NUM_COLUMNS and c not in STAGE_COLUMNS and c not in BIN_FLAG_COLUMNS 
]

print(f"Number of leftover columns: {len(leftover_cols)}")
print(leftover_cols, df_og.shape)  # preview first 50

def make_cost_stratum_3class(df):
    # Default to low-cost (class 0)
    cost_stratum = pd.Series(0, index=df.index)
    cost_stratum[(df['highcost_gt_50000'] == 1) & (df['highcost_gt_100000'] == 0)] = 1
    # Emergent high cost (class 2): 100k to 200k
    cost_stratum[(df['highcost_gt_100000'] == 1) & (df['highcost_gt_200000'] == 0)] = 2
    # High cost (class 2): 200k+
    cost_stratum[df['highcost_gt_200000'] == 1] = 3
    return cost_stratum

# Add the new column to your data
df_og['cost_stratum_2018'] = make_cost_stratum_3class(df_og)
print(df_og["cost_stratum_2018"].value_counts(dropna=False))

cutoff_columns = [col for col in df_og.columns if col.startswith('highcost_gt_')]

feature_cols = [c for c in df_og.columns
              if c not in  (['annual_cost_2017','annual_cost_2018_deflated',"ENROLID", "cost_stratum_2018"] 
              + cutoff_columns)]    # keep only predictors excl. cost of 2018 and cutoff of 2017
numeric_cols = df_og[feature_cols + ["cost_stratum_2018"]].select_dtypes(include=["number"]).columns
corrs = df_og[numeric_cols].corr()["cost_stratum_2018"].abs().sort_values(ascending=False)
# Columns to drop
high_corr_cols = corrs[corrs > 0.5].index.tolist()
# Remove the target column itself, if present
high_corr_cols = [col for col in high_corr_cols if col != "cost_stratum_2018"]
# Final filtered feature set
feature_cols = [col for col in feature_cols if col not in high_corr_cols]
print("High corr features dropped from prediction columns: ",high_corr_cols)
target_col = "highcost_gt_200000"
# Split data into train/test/val (same as multiobjective_bilevel.ipynb)
train_ids, test_ids, train_pd, test_pd = model_pipeline.train_test_split_enrol(
    df_og,
    target_col="cost_stratum_2018",
    test_size=0.3,
    verbose=False,
    random_state=123
)
print(f"Train shape: {train_pd.shape}, Test shape: {test_pd.shape}")
print("Feature cols:", len(feature_cols))

val_ids, test_ids, val_pd, test_pd = model_pipeline.train_test_split_enrol(
    test_pd, 
    target_col=target_col,
    test_size=0.5,
    verbose=False
)
X_test = test_pd[feature_cols]
y_test = test_pd[target_col]
X_val = val_pd[feature_cols]
y_val = val_pd[target_col]

print(f"Train: {train_pd.shape}, Val: {val_pd.shape}, Test: {test_pd.shape}")
print(f"Train target distribution:\n{train_pd[target_col].value_counts()}")

categorical cols:  ['ENROLID', 'INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
stage cols:  ['stage_2017', '2017Q1_max_ckd_stage', '2017Q2_max_ckd_stage', '2017Q3_max_ckd_stage', '2017Q4_max_ckd_stage']
['annual_cost_2017', 'highcost_gt_50000_2017', 'highcost_gt_75000_2017', 'highcost_gt_100000_2017', 'highcost_gt_200000_2017', 'highcost_gt_300000_2017', 'highcost_gt_400000_2017', 'highcost_gt_500000_2017', 'annual_cost_2018_deflated', 'highcost_gt_50000', 'highcost_gt_75000', 'highcost_gt_100000', 'highcost_gt_200000', 'highcost_gt_300000', 'highcost_gt_400000', 'highcost_gt_500000', '2017Q1_ckd_cost', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4

In [7]:
# ============================================================================
# COMPETING UNDERSAMPLING METHODS: NearMiss, Tomek Links, ENN, NCL
# ============================================================================

from imblearn.under_sampling import NearMiss, TomekLinks, EditedNearestNeighbours, NeighbourhoodCleaningRule
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# OCT hyperparameters (same as other scripts)
OCT_DEPTHS = [7, 9]
OCT_MINBUCKETS = [50, 100, 120, 150]
OCT_CPS = [0.00001, 0.0001, 0.001, 0.01]

# Prepare training data
X_train_raw = train_pd[feature_cols].copy()
y_train = train_pd[target_col].copy()

print("="*80)
print("ORIGINAL TRAINING DATA")
print("="*80)
print(f"Shape: {X_train_raw.shape}")
print(f"Target distribution:")
print(y_train.value_counts())
print(f"Class ratio: {(y_train == 0).sum()}:{(y_train == 1).sum()} (majority:minority)")

# Preprocess features for distance-based methods
# These methods need numeric features, so we'll use the preprocessor
from model_pipeline import get_preprocessor
preprocessor_train = get_preprocessor(
    df=X_train_raw,
    categorical_cols=CAT_COLUMNS,
    numeric_cols=TRUE_NUM_COLUMNS,
    verbose=False
)
X_train_processed = preprocessor_train.fit_transform(X_train_raw)
print(f"\nProcessed training data shape: {X_train_processed.shape}")


ORIGINAL TRAINING DATA
Shape: (23479, 78)
Target distribution:
highcost_gt_200000
0    22819
1      660
Name: count, dtype: int64
Class ratio: 22819:660 (majority:minority)

Processed training data shape: (23479, 90)


## APPLY UNDERSAMPLING METHODS

In [4]:
# ============================================================================
# APPLY UNDERSAMPLING METHODS
# ============================================================================
# Note: ENN and NCL are cleaning methods (remove noisy samples), not aggressive undersamplers.
# They typically remove only a small percentage of majority samples.

def reconstruct_dataframe_from_resampled(X_res, y_res, X_original_processed, train_df_original):
    """Reconstruct dataframe from resampled data by matching to original indices."""
    # Convert resampled arrays to DataFrames for comparison
    X_res_df = pd.DataFrame(X_res)
    X_orig_df = pd.DataFrame(X_original_processed)
    
    # Create a mapping by matching rows
    # This is approximate but works for most cases
    matching_indices = []
    for idx in range(len(X_res_df)):
        row = X_res_df.iloc[idx]
        # Find matching rows in original (use first match)
        matches = (X_orig_df == row).all(axis=1)
        if matches.any():
            matching_indices.append(X_orig_df.index[matches][0])
        else:
            # If no exact match, find closest (using hamming distance on boolean)
            # Fallback: use first available index (shouldn't happen often)
            pass
    
    # Alternative: if sample_indices_ is available, use it
    # Otherwise, match by row values
    if len(matching_indices) == len(X_res):
        return train_df_original.iloc[matching_indices].reset_index(drop=True)
    else:
        # Fallback: create new dataframe from resampled data
        # This won't have all original columns, so we'll use a different approach
        raise ValueError("Could not reconstruct dataframe from resampled indices")

undersampled_datasets = {}
undersampling_methods = {}

print("="*80)
print("APPLYING UNDERSAMPLING METHODS")
print("="*80)
print("\nNOTE: ENN and NCL are cleaning methods (remove noisy samples), not aggressive undersamplers.")
print("      They typically remove only 1-5% of majority samples.")

# Helper function to get indices from resampled data
def get_indices_from_resampler(resampler, X_original, y_original, X_resampled, y_resampled):
    """Get original indices from resampler."""
    # Try to use sample_indices_ if available
    if hasattr(resampler, 'sample_indices_') and resampler.sample_indices_ is not None:
        return resampler.sample_indices_
    
    # Otherwise, match rows to find indices
    # Create indices list by matching resampled rows to original
    indices = []
    X_orig_df = pd.DataFrame(X_original)
    X_res_df = pd.DataFrame(X_resampled)
    
    for idx_res in range(len(X_res_df)):
        row_res = X_res_df.iloc[idx_res].values
        # Find matching row in original (tolerance for floating point)
        matches = np.allclose(X_orig_df.values, row_res, atol=1e-6, axis=1)
        if matches.any():
            # Take first match
            orig_idx = X_orig_df.index[matches][0]
            if orig_idx not in indices:
                indices.append(orig_idx)
        else:
            # Find closest match using Euclidean distance
            distances = np.linalg.norm(X_orig_df.values - row_res, axis=1)
            closest_idx = X_orig_df.index[np.argmin(distances)]
            if closest_idx not in indices:
                indices.append(closest_idx)
    
    return indices

# 1. NearMiss-1: Select majority samples closest to minority samples
print("\n1. NearMiss-1: Selecting majority samples closest to minority...")
try:
    nm1 = NearMiss(version=1, n_neighbors=3)  # Removed random_state
    X_res_nm1, y_res_nm1 = nm1.fit_resample(X_train_processed, y_train.values)
    
    # Get original indices
    indices_nm1 = get_indices_from_resampler(nm1, X_train_processed, y_train.values, X_res_nm1, y_res_nm1)
    undersampled_nm1 = train_pd.iloc[indices_nm1].reset_index(drop=True)
    
    undersampled_datasets['NearMiss-1'] = undersampled_nm1
    undersampling_methods['NearMiss-1'] = nm1
    
    print(f"  ✓ NearMiss-1: {len(undersampled_nm1):,} samples")
    print(f"    Distribution: {undersampled_nm1[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_nm1):,} ({len(undersampled_nm1)/len(train_pd)*100:.1f}%)")
except Exception as e:
    print(f"  ✗ Error with NearMiss-1: {e}")
    import traceback
    traceback.print_exc()

# 2. NearMiss-2: Select majority samples closest to farthest minority samples
print("\n2. NearMiss-2: Selecting majority samples closest to farthest minority...")
try:
    nm2 = NearMiss(version=2, n_neighbors=3)  # Removed random_state
    X_res_nm2, y_res_nm2 = nm2.fit_resample(X_train_processed, y_train.values)
    
    indices_nm2 = get_indices_from_resampler(nm2, X_train_processed, y_train.values, X_res_nm2, y_res_nm2)
    undersampled_nm2 = train_pd.iloc[indices_nm2].reset_index(drop=True)
    
    undersampled_datasets['NearMiss-2'] = undersampled_nm2
    undersampling_methods['NearMiss-2'] = nm2
    
    print(f"  ✓ NearMiss-2: {len(undersampled_nm2):,} samples")
    print(f"    Distribution: {undersampled_nm2[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_nm2):,} ({len(undersampled_nm2)/len(train_pd)*100:.1f}%)")
except Exception as e:
    print(f"  ✗ Error with NearMiss-2: {e}")
    import traceback
    traceback.print_exc()

# 3. NearMiss-3: Select majority samples with minimum average distance to k nearest minority samples
print("\n3. NearMiss-3: Selecting majority with min avg distance to k nearest minority...")
try:
    nm3 = NearMiss(version=3, n_neighbors=3)  # Removed random_state
    X_res_nm3, y_res_nm3 = nm3.fit_resample(X_train_processed, y_train.values)
    
    indices_nm3 = get_indices_from_resampler(nm3, X_train_processed, y_train.values, X_res_nm3, y_res_nm3)
    undersampled_nm3 = train_pd.iloc[indices_nm3].reset_index(drop=True)
    
    undersampled_datasets['NearMiss-3'] = undersampled_nm3
    undersampling_methods['NearMiss-3'] = nm3
    
    print(f"  ✓ NearMiss-3: {len(undersampled_nm3):,} samples")
    print(f"    Distribution: {undersampled_nm3[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_nm3):,} ({len(undersampled_nm3)/len(train_pd)*100:.1f}%)")
except Exception as e:
    print(f"  ✗ Error with NearMiss-3: {e}")
    import traceback
    traceback.print_exc()

# 4. Tomek Links: Remove Tomek link pairs (noise cleaning - minimal undersampling)
print("\n4. Tomek Links: Removing Tomek link pairs (noise cleaning)...")
print("   NOTE: Tomek Links is a cleaning method - it removes only noisy samples, not aggressive undersampling")
try:
    tomek = TomekLinks(sampling_strategy='majority')  # Removed random_state
    X_res_tomek, y_res_tomek = tomek.fit_resample(X_train_processed, y_train.values)
    
    indices_tomek = get_indices_from_resampler(tomek, X_train_processed, y_train.values, X_res_tomek, y_res_tomek)
    undersampled_tomek = train_pd.iloc[indices_tomek].reset_index(drop=True)
    
    undersampled_datasets['TomekLinks'] = undersampled_tomek
    undersampling_methods['TomekLinks'] = tomek
    
    print(f"  ✓ Tomek Links: {len(undersampled_tomek):,} samples")
    print(f"    Distribution: {undersampled_tomek[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_tomek):,} ({len(undersampled_tomek)/len(train_pd)*100:.1f}%)")
except Exception as e:
    print(f"  ✗ Error with Tomek Links: {e}")
    import traceback
    traceback.print_exc()

# 5. Edited Nearest Neighbors (ENN): Remove majority samples misclassified by k-NN (cleaning)
print("\n5. Edited Nearest Neighbors (ENN): Removing misclassified majority samples...")
print("   NOTE: ENN is a cleaning method - removes noisy samples, typically 1-5% reduction")
try:
    enn = EditedNearestNeighbours(sampling_strategy='majority', n_neighbors=3, kind_sel='all')
    X_res_enn, y_res_enn = enn.fit_resample(X_train_processed, y_train.values)
    
    indices_enn = get_indices_from_resampler(enn, X_train_processed, y_train.values, X_res_enn, y_res_enn)
    undersampled_enn = train_pd.iloc[indices_enn].reset_index(drop=True)
    
    undersampled_datasets['ENN'] = undersampled_enn
    undersampling_methods['ENN'] = enn
    
    print(f"  ✓ ENN: {len(undersampled_enn):,} samples")
    print(f"    Distribution: {undersampled_enn[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_enn):,} ({len(undersampled_enn)/len(train_pd)*100:.1f}%)")
except Exception as e:
    print(f"  ✗ Error with ENN: {e}")
    import traceback
    traceback.print_exc()

# 6. Neighborhood Cleaning Rule (NCL): More aggressive cleaning than ENN (but still cleaning)
print("\n6. Neighborhood Cleaning Rule (NCL): More aggressive cleaning than ENN...")
print("   NOTE: NCL is a cleaning method - removes more noisy samples than ENN, but still limited reduction")
try:
    ncl = NeighbourhoodCleaningRule(sampling_strategy='majority', n_neighbors=3, kind_sel='all')
    X_res_ncl, y_res_ncl = ncl.fit_resample(X_train_processed, y_train.values)
    
    indices_ncl = get_indices_from_resampler(ncl, X_train_processed, y_train.values, X_res_ncl, y_res_ncl)
    undersampled_ncl = train_pd.iloc[indices_ncl].reset_index(drop=True)
    
    undersampled_datasets['NCL'] = undersampled_ncl
    undersampling_methods['NCL'] = ncl
    
    print(f"  ✓ NCL: {len(undersampled_ncl):,} samples")
    print(f"    Distribution: {undersampled_ncl[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_ncl):,} ({len(undersampled_ncl)/len(train_pd)*100:.1f}%)")
except Exception as e:
    print(f"  ✗ Error with NCL: {e}")
    import traceback
    traceback.print_exc()

# 7. Random UnderSampler (for comparison - aggressive 1:1 ratio)
print("\n7. Random UnderSampler: Random undersampling to 1:1 ratio (for comparison)...")
from imblearn.under_sampling import RandomUnderSampler
try:
    rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
    X_res_rus, y_res_rus = rus.fit_resample(X_train_processed, y_train.values)
    
    indices_rus = get_indices_from_resampler(rus, X_train_processed, y_train.values, X_res_rus, y_res_rus)
    undersampled_rus = train_pd.iloc[indices_rus].reset_index(drop=True)
    
    undersampled_datasets['RandomUnderSampler'] = undersampled_rus
    undersampling_methods['RandomUnderSampler'] = rus
    
    print(f"  ✓ Random UnderSampler: {len(undersampled_rus):,} samples")
    print(f"    Distribution: {undersampled_rus[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_rus):,} ({len(undersampled_rus)/len(train_pd)*100:.1f}%)")
except Exception as e:
    print(f"  ✗ Error with Random UnderSampler: {e}")
    import traceback
    traceback.print_exc()

print(f"\n{'='*80}")
print(f"Successfully created {len(undersampled_datasets)} undersampled datasets")
print("="*80)
print("\nSummary:")
print(f"{'Method':<20s} {'Samples':>10s} {'Reduction':>12s} {'Minority':>10s} {'Majority':>10s}")
print("-" * 70)
for method, df in undersampled_datasets.items():
    reduction_pct = (1 - len(df)/len(train_pd)) * 100
    n_minority = (df[target_col] == 1).sum()
    n_majority = (df[target_col] == 0).sum()
    print(f"{method:<20s} {len(df):>10,} {reduction_pct:>11.1f}% {n_minority:>10,} {n_majority:>10,}")

# Create directory for saving predictions
predictions_dir = "competing_methods_results/predictions"
os.makedirs(predictions_dir, exist_ok=True)

APPLYING UNDERSAMPLING METHODS

NOTE: ENN and NCL are cleaning methods (remove noisy samples), not aggressive undersamplers.
      They typically remove only 1-5% of majority samples.

1. NearMiss-1: Selecting majority samples closest to minority...
  ✓ NearMiss-1: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)

2. NearMiss-2: Selecting majority samples closest to farthest minority...
  ✓ NearMiss-2: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)

3. NearMiss-3: Selecting majority with min avg distance to k nearest minority...
  ✓ NearMiss-3: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)

4. Tomek Links: Removing Tomek link pairs (noise cleaning)...
   NOTE: Tomek Links is a cleaning method - it removes only noisy samples, not aggressive undersampling
  ✓ Tomek Links: 23,333 samples
    Distribution: {0: 22673, 1: 660}
    Reduction: 23,479 → 23,333 (99.4%)

5. Edited N

## Train a bunch of random seed undersampler

In [11]:
# 7. Random UnderSampler (for comparison - aggressive 1:1 ratio)
print("\n7. Random UnderSampler: Random undersampling to 1:1 ratio (for comparison)...")
from imblearn.under_sampling import RandomUnderSampler
random_undersampled_datasets = {}
random_undersampling_methods = {}
for seed in range(10):
    rus = RandomUnderSampler(sampling_strategy='auto', random_state=seed)
    X_res_rus, y_res_rus = rus.fit_resample(X_train_processed, y_train.values)

    indices_rus = get_indices_from_resampler(rus, X_train_processed, y_train.values, X_res_rus, y_res_rus)
    undersampled_rus = train_pd.iloc[indices_rus].reset_index(drop=True)
    
    random_undersampled_datasets['RandomUnderSampler_seed_'+str(seed)] = undersampled_rus
    random_undersampling_methods['RandomUnderSampler_seed_'+str(seed)] = rus
    
    print(f"  ✓ Random UnderSampler: {len(undersampled_rus):,} samples")
    print(f"    Distribution: {undersampled_rus[target_col].value_counts().to_dict()}")
    print(f"    Reduction: {len(train_pd):,} → {len(undersampled_rus):,} ({len(undersampled_rus)/len(train_pd)*100:.1f}%)")
    
# ============================================================================
# TRAIN AND EVALUATE OCT MODELS ON EACH UNDERSAMPLED DATASET
# ============================================================================

print("="*80)
print("TRAINING OCT MODELS ON UNDERSAMPLED DATASETS")
print("="*80)

random_avg_results = []

for method_name, undersampled_df in random_undersampled_datasets.items():
    print(f"\n{'='*80}")
    print(f"METHOD: {method_name}")
    print(f"{'='*80}")
    
    n_samples = len(undersampled_df)
    n_minority = (undersampled_df[target_col] == 1).sum()
    n_majority = (undersampled_df[target_col] == 0).sum()
    
    print(f"Dataset size: {n_samples:,} samples")
    print(f"  Minority: {n_minority:,}")
    print(f"  Majority: {n_majority:,}")
    print(f"  Ratio: {n_majority/n_minority:.2f}:1")
    

    # Train OCT model
    print(f"\n  Training OCT model...")
    balanced_model, balanced_params, _, preprocessor, feature_names = finetune_oct(
        X_train=undersampled_df[feature_cols],
        y_train=undersampled_df[target_col],
        X_val=X_val,
        y_val=y_val,
        categorical_cols=CAT_COLUMNS,
        numeric_cols=TRUE_NUM_COLUMNS,
        depths=OCT_DEPTHS,
        minbuckets=OCT_MINBUCKETS,
        cps=OCT_CPS,
    )
    
    print(f"    ✓ Training complete")
    print(f"    Best params: {balanced_params}")
    
    # Evaluate on test set and save predictions
    print(f"  Evaluating on test set...")
    # Save predictions to method-specific directory
    method_results_dir = f"competing_methods_results/{method_name}"
    metrics = evaluate_binary_oct(
        balanced_model, X_test, y_test, preprocessor, feature_names,
        results_dir=method_results_dir, ratio=None
    )
    
    # Store results
    result_row = {
        'method': method_name,
        'n_train_samples': n_samples,
        'n_train_minority': n_minority,
        'n_train_majority': n_majority,
        'best_depth': balanced_params[0] if isinstance(balanced_params, tuple) else None,
        'best_minbucket': balanced_params[1] if isinstance(balanced_params, tuple) else None,
        'best_cp': balanced_params[2] if isinstance(balanced_params, tuple) else None,
    }
    
    if isinstance(metrics, dict):
        result_row.update(metrics)
        print(f"    ✓ Evaluation complete")
        print(f"      AUC: {metrics.get('auc', 'N/A'):.4f}" if isinstance(metrics.get('auc'), (int, float)) else f"      AUC: {metrics.get('auc', 'N/A')}")
        print(f"      PR-AUC: {metrics.get('pr_auc', 'N/A'):.4f}" if isinstance(metrics.get('pr_auc'), (int, float)) else f"      PR-AUC: {metrics.get('pr_auc', 'N/A')}")
        print(f"      MCC: {metrics.get('best_mcc', 'N/A'):.4f}" if isinstance(metrics.get('best_mcc'), (int, float)) else f"      MCC: {metrics.get('best_mcc', 'N/A')}")
        print(f"      Recall (G-mean): {metrics.get('balanced_recall_gmean', 'N/A'):.4f}" if isinstance(metrics.get('balanced_recall_gmean'), (int, float)) else f"      Recall (G-mean): {metrics.get('balanced_recall_gmean', 'N/A')}")
    else:
        result_row['error'] = str(metrics)
        print(f"    ✗ Evaluation error: {metrics}")
    
    random_avg_results.append(result_row)
            
random_avg_results


7. Random UnderSampler: Random undersampling to 1:1 ratio (for comparison)...
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 samples
    Distribution: {0: 660, 1: 660}
    Reduction: 23,479 → 1,320 (5.6%)
  ✓ Random UnderSampler: 1,320 sa

[{'method': 'RandomUnderSampler_seed_0',
  'n_train_samples': 1320,
  'n_train_minority': np.int64(660),
  'n_train_majority': np.int64(660),
  'best_depth': 7,
  'best_minbucket': 150,
  'best_cp': 1e-05,
  'auc': 0.8134040438952734,
  'pr_auc': np.float64(0.09043367194564267),
  'best_mcc': 0.2345594885184981,
  'best_mcc_threshold': 0.5398789346246974,
  'recall_mcc': 0.7394366197183099,
  'precision_mcc': 0.10725229826353422,
  'optimal_f1': 0.18733273860410127,
  'balanced_recall_gmean': 0.7394366197183099,
  'balanced_specificity_gmean': 0.8212678936605317,
  'precision_gmean': 0.10725229826353422},
 {'method': 'RandomUnderSampler_seed_1',
  'n_train_samples': 1320,
  'n_train_minority': np.int64(660),
  'n_train_majority': np.int64(660),
  'best_depth': 7,
  'best_minbucket': 50,
  'best_cp': 0.01,
  'auc': 0.8019060168783663,
  'pr_auc': np.float64(0.08353223742808139),
  'best_mcc': 0.22232698069332915,
  'best_mcc_threshold': 0.5540404040404041,
  'recall_mcc': 0.746478873239

## TRAIN AND EVALUATE OCT MODELS ON EACH UNDERSAMPLED DATASET

In [5]:
# ============================================================================
# TRAIN AND EVALUATE OCT MODELS ON EACH UNDERSAMPLED DATASET
# ============================================================================

print("="*80)
print("TRAINING OCT MODELS ON UNDERSAMPLED DATASETS")
print("="*80)

all_results = []

for method_name, undersampled_df in undersampled_datasets.items():
    print(f"\n{'='*80}")
    print(f"METHOD: {method_name}")
    print(f"{'='*80}")
    
    n_samples = len(undersampled_df)
    n_minority = (undersampled_df[target_col] == 1).sum()
    n_majority = (undersampled_df[target_col] == 0).sum()
    
    print(f"Dataset size: {n_samples:,} samples")
    print(f"  Minority: {n_minority:,}")
    print(f"  Majority: {n_majority:,}")
    print(f"  Ratio: {n_majority/n_minority:.2f}:1")
    
    try:
        # Train OCT model
        print(f"\n  Training OCT model...")
        balanced_model, balanced_params, _, preprocessor, feature_names = finetune_oct(
            X_train=undersampled_df[feature_cols],
            y_train=undersampled_df[target_col],
            X_val=X_val,
            y_val=y_val,
            categorical_cols=CAT_COLUMNS,
            numeric_cols=TRUE_NUM_COLUMNS,
            depths=OCT_DEPTHS,
            minbuckets=OCT_MINBUCKETS,
            cps=OCT_CPS,
        )
        
        print(f"    ✓ Training complete")
        print(f"    Best params: {balanced_params}")
        
        # Evaluate on test set and save predictions
        print(f"  Evaluating on test set...")
        # Save predictions to method-specific directory
        method_results_dir = f"competing_methods_results/{method_name}"
        metrics = evaluate_binary_oct(
            balanced_model, X_test, y_test, preprocessor, feature_names,
            results_dir=method_results_dir, ratio=None
        )
        
        # Store results
        result_row = {
            'method': method_name,
            'n_train_samples': n_samples,
            'n_train_minority': n_minority,
            'n_train_majority': n_majority,
            'best_depth': balanced_params[0] if isinstance(balanced_params, tuple) else None,
            'best_minbucket': balanced_params[1] if isinstance(balanced_params, tuple) else None,
            'best_cp': balanced_params[2] if isinstance(balanced_params, tuple) else None,
        }
        
        if isinstance(metrics, dict):
            result_row.update(metrics)
            print(f"    ✓ Evaluation complete")
            print(f"      AUC: {metrics.get('auc', 'N/A'):.4f}" if isinstance(metrics.get('auc'), (int, float)) else f"      AUC: {metrics.get('auc', 'N/A')}")
            print(f"      PR-AUC: {metrics.get('pr_auc', 'N/A'):.4f}" if isinstance(metrics.get('pr_auc'), (int, float)) else f"      PR-AUC: {metrics.get('pr_auc', 'N/A')}")
            print(f"      MCC: {metrics.get('best_mcc', 'N/A'):.4f}" if isinstance(metrics.get('best_mcc'), (int, float)) else f"      MCC: {metrics.get('best_mcc', 'N/A')}")
            print(f"      Recall (G-mean): {metrics.get('balanced_recall_gmean', 'N/A'):.4f}" if isinstance(metrics.get('balanced_recall_gmean'), (int, float)) else f"      Recall (G-mean): {metrics.get('balanced_recall_gmean', 'N/A')}")
        else:
            result_row['error'] = str(metrics)
            print(f"    ✗ Evaluation error: {metrics}")
        
        all_results.append(result_row)
        
    except Exception as e:
        print(f"    ✗ ERROR: {e}")
        import traceback
        traceback.print_exc()
        
        all_results.append({
            'method': method_name,
            'n_train_samples': n_samples,
            'error': str(e)
        })

print(f"\n{'='*80}")
print("ALL MODELS TRAINED AND EVALUATED")
print("="*80)


TRAINING OCT MODELS ON UNDERSAMPLED DATASETS

METHOD: NearMiss-1
Dataset size: 1,320 samples
  Minority: 660
  Majority: 660
  Ratio: 1.00:1

  Training OCT model...
Finetuning OCT with depths: [7, 9], minbuckets: [50, 100, 120, 150], cps: [1e-05, 0.0001, 0.001, 0.01], for best PR-AUC!!!
→ Building preprocessor:
   • OneHotEncoder on: ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
   • StandardScaler on: ['2017Q1_ckd_cost', '2017Q1_ckd_claims', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_ckd_claims', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_ckd_claims', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4_ckd_claims', '2017Q4_direct_ckd_cost', '2017Q4_procedure_ckd_cost', '2017Q4_comorbidity_ckd_cost', 'ckd_cost_trend_20

In [8]:
# ============================================================================
# COMPARISON: RESULTS SUMMARY AND RANKINGS
# ============================================================================

results_df = pd.DataFrame(all_results)

print("="*80)
print("COMPARISON OF COMPETING METHODS")
print("="*80)

# Display results table
if len(results_df) > 0:
    display_cols = ['method', 'n_train_samples', 'n_train_minority', 'n_train_majority']
    metric_cols = ['auc', 'pr_auc', 'best_mcc', 'balanced_recall_gmean', 'balanced_specificity_gmean', 'optimal_f1']
    
    for col in metric_cols:
        if col in results_df.columns:
            display_cols.append(col)
    
    display_cols = [c for c in display_cols if c in results_df.columns]
    
    # Format numeric columns
    for col in metric_cols:
        if col in results_df.columns:
            results_df[col] = results_df[col].apply(
                lambda x: f"{x:.4f}" if pd.notna(x) and isinstance(x, (int, float)) else "N/A"
            )
    
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    pd.set_option('display.max_colwidth', 30)
    
    print("\n" + "="*80)
    print("ALL METHODS - COMPLETE RESULTS")
    print("="*80)
    print(results_df[display_cols].to_string(index=False))
    
    # Rankings by different metrics
    if 'auc' in results_df.columns:
        print("\n" + "="*80)
        print("RANKING BY AUC (descending)")
        print("="*80)
        # Convert back to numeric for sorting
        results_df['auc_numeric'] = pd.to_numeric(results_df['auc'], errors='coerce')
        sorted_by_auc = results_df.sort_values('auc_numeric', ascending=False)
        print(sorted_by_auc[display_cols].to_string(index=False))
        
        if 'best_mcc' in results_df.columns:
            print("\n" + "="*80)
            print("RANKING BY MCC (descending)")
            print("="*80)
            results_df['mcc_numeric'] = pd.to_numeric(results_df['best_mcc'], errors='coerce')
            sorted_by_mcc = results_df.sort_values('mcc_numeric', ascending=False)
            print(sorted_by_mcc[display_cols].to_string(index=False))
        
        if 'pr_auc' in results_df.columns:
            print("\n" + "="*80)
            print("RANKING BY PR-AUC (descending)")
            print("="*80)
            results_df['pr_auc_numeric'] = pd.to_numeric(results_df['pr_auc'], errors='coerce')
            sorted_by_pr_auc = results_df.sort_values('pr_auc_numeric', ascending=False)
            print(sorted_by_pr_auc[display_cols].to_string(index=False))
    
    # Save results
    output_dir = "competing_methods_results"
    os.makedirs(output_dir, exist_ok=True)
    results_path = f"{output_dir}/competing_methods_metrics.csv"
    results_df.to_csv(results_path, index=False)
    print(f"\n{'='*80}")
    print(f"✓ Results saved to: {results_path}")
    print("="*80)
else:
    print("⚠️ No results to display")


COMPARISON OF COMPETING METHODS

ALL METHODS - COMPLETE RESULTS
            method  n_train_samples  n_train_minority  n_train_majority    auc pr_auc best_mcc balanced_recall_gmean balanced_specificity_gmean optimal_f1
        NearMiss-1             1320               660               660 0.7851 0.0789   0.1937                0.8380                     0.6607     0.1628
        NearMiss-2             1320               660               660 0.3899 0.0229   0.0000                0.8099                     0.0521     0.0549
        NearMiss-3             1320               660               660 0.2299 0.0183   0.0000                0.2324                     0.2824     0.0549
        TomekLinks            23333               660             22673 0.6173 0.1141   0.2402                0.2535                     0.9800     0.2609
               ENN            22355               660             21695 0.6232 0.0992   0.2422                0.2676                     0.9779     0.2639
      

## SYMMETRIC LEAF EVALUATION: Compare each competing method against vanilla OCT
============================================================================


In [ ]:
y_test_series = pd.Series(y_test.values, index=test_pd["ENROLID"])
subgroup_results = symmetric_leaf_evaluation_oct(
    mv_pred_path=vanilla_pred_path,
    ms_pred_path=balanced_pred_path,
    y_test=y_test_series,
    enrolid_col="ENROLID",
)


In [ ]:

import importlib
import symmetric_excess_AUC
importlib.reload(symmetric_excess_AUC)
from symmetric_excess_AUC import symmetric_leaf_evaluation_oct
import glob

print("="*80)
print("SYMMETRIC LEAF EVALUATION")
print("="*80)
print("Comparing each competing method against vanilla OCT baseline...")

# Path to vanilla OCT predictions
vanilla_pred_path = "vanilla_oct/predictions/oct_predictions.csv"

# Dictionary to store all symmetric evaluation results
symmetric_results = {}

# Helper function to safely format numeric values
def format_val(val, fmt='.6f'):
    if val is None or (isinstance(val, str) and val == 'N/A'):
        return 'N/A'
    try:
        if isinstance(val, (int, float)):
            return f"{val:{fmt}}"
        return str(val)
    except (ValueError, TypeError):
        return str(val)

# Discover all methods from competing_methods_results folder
competing_methods_dir = "competing_methods_results"
all_method_dirs = []

if os.path.exists(competing_methods_dir):
    # Find all directories that contain predictions/oct_predictions.csv
    for item in os.listdir(competing_methods_dir):
        method_path = os.path.join(competing_methods_dir, item)
        pred_path = os.path.join(method_path, "predictions", "oct_predictions.csv")
        if os.path.isdir(method_path) and os.path.exists(pred_path):
            all_method_dirs.append(item)

all_method_dirs.sort()  # Sort for consistent ordering
print(f"Discovered {len(all_method_dirs)} methods from {competing_methods_dir}/")
print(f"Methods: {all_method_dirs}")

# Separate methods into regular methods and RandomUnderSampler seeds
regular_methods = []
random_seed_methods = []

for method_name in all_method_dirs:
    if method_name.startswith('RandomUnderSampler_seed_'):
        random_seed_methods.append(method_name)
    else:
        regular_methods.append(method_name)

# Process regular methods (non-RandomUnderSampler seeds)
for method_name in regular_methods:
    print(f"\n{'='*80}")
    print(f"METHOD: {method_name}")
    print(f"{'='*80}")
    
    # Path to this method's predictions
    balanced_pred_path = f"competing_methods_results/{method_name}/predictions/oct_predictions.csv"
    
    try:
        # Check if predictions file exists
        if not os.path.exists(balanced_pred_path):
            print(f"  ⚠️  Predictions file not found: {balanced_pred_path}")
            print(f"     Skipping symmetric evaluation for {method_name}")
            continue
        
        if not os.path.exists(vanilla_pred_path):
            print(f"  ⚠️  Vanilla predictions file not found: {vanilla_pred_path}")
            print(f"     Please ensure vanilla OCT baseline was trained successfully")
            continue
        
        # Calculate symmetric leaf evaluation
        subgroup_results = symmetric_leaf_evaluation_oct(
            mv_pred_path=vanilla_pred_path,
            ms_pred_path=balanced_pred_path,
            y_test=y_test,
        )
        
        # Store results
        symmetric_results[method_name] = subgroup_results
        
        # Display results
        print(f"\n  Symmetric Excess AUC Results:")
        if 'scores' in subgroup_results:
            scores = subgroup_results['scores']
            
            print(f"    Excess ROC (balanced|vanilla): {format_val(scores.get('excess_ROC_s|v', 'N/A'))}")
            print(f"    Excess ROC (vanilla|balanced): {format_val(scores.get('excess_ROC_v|s', 'N/A'))}")
            print(f"    Symmetric Excess ROC: {format_val(scores.get('sym_excess_ROC', 'N/A'))}")
            print(f"    Coverage (informative vanilla leaves): {format_val(scores.get('coverage_informative_v', 'N/A'), '.4f')}")
            print(f"    Coverage (informative balanced leaves): {format_val(scores.get('coverage_informative_s', 'N/A'), '.4f')}")
        
        # Also display global metrics from overall if available
        if 'overall' in subgroup_results:
            overall = subgroup_results['overall']
            print(f"\n  Global Metrics:")
            print(f"    Vanilla OCT ROC-AUC: {format_val(overall.get('M_v_global_ROC', 'N/A'), '.6f')}")
            print(f"    Balanced OCT ROC-AUC: {format_val(overall.get('M_s_global_ROC', 'N/A'), '.6f')}")
            print(f"    Vanilla OCT PR-AUC: {format_val(overall.get('M_v_global_PR', 'N/A'), '.6f')}")
            print(f"    Balanced OCT PR-AUC: {format_val(overall.get('M_s_global_PR', 'N/A'), '.6f')}")
        
        if 'ci' in subgroup_results:
            print(f"\n  Bootstrap Confidence Intervals (95%):")
            ci = subgroup_results['ci']
            if 'excess_ROC_s|v' in ci:
                ci_s_v = ci['excess_ROC_s|v']
                lo = format_val(ci_s_v.get('lo', 'N/A'))
                hi = format_val(ci_s_v.get('hi', 'N/A'))
                print(f"    Excess ROC (balanced|vanilla): [{lo}, {hi}]")
            if 'excess_ROC_v|s' in ci:
                ci_v_s = ci['excess_ROC_v|s']
                lo = format_val(ci_v_s.get('lo', 'N/A'))
                hi = format_val(ci_v_s.get('hi', 'N/A'))
                print(f"    Excess ROC (vanilla|balanced): [{lo}, {hi}]")
        
    except Exception as e:
        print(f"  ✗ ERROR with symmetric evaluation for {method_name}: {e}")
        import traceback
        traceback.print_exc()

# Process RandomUnderSampler seeds separately and compute averages
if len(random_seed_methods) > 0:
    print(f"\n{'='*80}")
    print("RANDOMUNDERSAMPLER SEEDS: Individual Evaluations")
    print("="*80)
    
    random_seed_results = {}
    
    for method_name in random_seed_methods:
        print(f"\n{'='*80}")
        print(f"METHOD: {method_name}")
        print(f"{'='*80}")
        
        # Path to this method's predictions
        balanced_pred_path = f"competing_methods_results/{method_name}/predictions/oct_predictions.csv"
        
        try:
            # Check if predictions file exists
            if not os.path.exists(balanced_pred_path):
                print(f"  ⚠️  Predictions file not found: {balanced_pred_path}")
                print(f"     Skipping symmetric evaluation for {method_name}")
                continue
            
            if not os.path.exists(vanilla_pred_path):
                print(f"  ⚠️  Vanilla predictions file not found: {vanilla_pred_path}")
                print(f"     Please ensure vanilla OCT baseline was trained successfully")
                continue
            
            # Calculate symmetric leaf evaluation
            subgroup_results = symmetric_leaf_evaluation_oct(
                mv_pred_path=vanilla_pred_path,
                ms_pred_path=balanced_pred_path,
                y_test=y_test,
            )
            
            # Store results
            random_seed_results[method_name] = subgroup_results
            symmetric_results[method_name] = subgroup_results
            
            # Display results
            print(f"\n  Symmetric Excess AUC Results:")
            if 'scores' in subgroup_results:
                scores = subgroup_results['scores']
                
                print(f"    Excess ROC (balanced|vanilla): {format_val(scores.get('excess_ROC_s|v', 'N/A'))}")
                print(f"    Excess ROC (vanilla|balanced): {format_val(scores.get('excess_ROC_v|s', 'N/A'))}")
                print(f"    Symmetric Excess ROC: {format_val(scores.get('sym_excess_ROC', 'N/A'))}")
                print(f"    Coverage (informative vanilla leaves): {format_val(scores.get('coverage_informative_v', 'N/A'), '.4f')}")
                print(f"    Coverage (informative balanced leaves): {format_val(scores.get('coverage_informative_s', 'N/A'), '.4f')}")
            
            if 'ci' in subgroup_results:
                print(f"\n  Bootstrap Confidence Intervals (95%):")
                ci = subgroup_results['ci']
                if 'excess_ROC_s|v' in ci:
                    ci_s_v = ci['excess_ROC_s|v']
                    lo = format_val(ci_s_v.get('lo', 'N/A'))
                    hi = format_val(ci_s_v.get('hi', 'N/A'))
                    print(f"    Excess ROC (balanced|vanilla): [{lo}, {hi}]")
                if 'excess_ROC_v|s' in ci:
                    ci_v_s = ci['excess_ROC_v|s']
                    lo = format_val(ci_v_s.get('lo', 'N/A'))
                    hi = format_val(ci_v_s.get('hi', 'N/A'))
                    print(f"    Excess ROC (vanilla|balanced): [{lo}, {hi}]")
            
        except Exception as e:
            print(f"  ✗ ERROR with symmetric evaluation for {method_name}: {e}")
            import traceback
            traceback.print_exc()
    
    # Calculate average metrics across all RandomUnderSampler seeds
    if len(random_seed_results) > 0:
        print(f"\n{'='*80}")
        print("RANDOMUNDERSAMPLER: AVERAGED METRICS ACROSS SEEDS")
        print("="*80)
        
        # Collect all metrics from seeds
        seed_metrics = []
        for method_name, results in random_seed_results.items():
            if 'scores' in results:
                scores = results['scores']
                seed_metrics.append({
                    'seed': method_name.replace('RandomUnderSampler_seed_', ''),
                    'excess_ROC_s|v': scores.get('excess_ROC_s|v', np.nan),
                    'excess_ROC_v|s': scores.get('excess_ROC_v|s', np.nan),
                    'sym_excess_ROC': scores.get('sym_excess_ROC', np.nan),
                    'coverage_v': scores.get('coverage_informative_v', np.nan),
                    'coverage_s': scores.get('coverage_informative_s', np.nan),
                })
        
        if seed_metrics:
            seed_df = pd.DataFrame(seed_metrics)
            
            # Calculate averages
            avg_metrics = {
                'method': 'RandomUnderSampler_AVG',
                'excess_ROC_s|v': seed_df['excess_ROC_s|v'].mean(),
                'excess_ROC_v|s': seed_df['excess_ROC_v|s'].mean(),
                'sym_excess_ROC': seed_df['sym_excess_ROC'].mean(),
                'coverage_v': seed_df['coverage_v'].mean(),
                'coverage_s': seed_df['coverage_s'].mean(),
            }
            
            # Calculate standard deviations
            std_metrics = {
                'method': 'RandomUnderSampler_STD',
                'excess_ROC_s|v': seed_df['excess_ROC_s|v'].std(),
                'excess_ROC_v|s': seed_df['excess_ROC_v|s'].std(),
                'sym_excess_ROC': seed_df['sym_excess_ROC'].std(),
                'coverage_v': seed_df['coverage_v'].std(),
                'coverage_s': seed_df['coverage_s'].std(),
            }
            
            print("\n  Individual Seed Results:")
            print(seed_df.to_string(index=False))
            
            print("\n  Averaged Metrics:")
            avg_df = pd.DataFrame([avg_metrics])
            print(avg_df.to_string(index=False))
            
            print("\n  Standard Deviations:")
            std_df = pd.DataFrame([std_metrics])
            print(std_df.to_string(index=False))
            
            # Store averaged results
            symmetric_results['RandomUnderSampler_AVG'] = {
                'scores': {
                    'excess_ROC_s|v': avg_metrics['excess_ROC_s|v'],
                    'excess_ROC_v|s': avg_metrics['excess_ROC_v|s'],
                    'sym_excess_ROC': avg_metrics['sym_excess_ROC'],
                    'coverage_informative_v': avg_metrics['coverage_v'],
                    'coverage_informative_s': avg_metrics['coverage_s'],
                }
            }



SYMMETRIC LEAF EVALUATION
Comparing each competing method against vanilla OCT baseline...
Discovered 17 methods from competing_methods_results/
Methods: ['ENN', 'NCL', 'NearMiss-1', 'NearMiss-2', 'NearMiss-3', 'RandomUnderSampler', 'RandomUnderSampler_seed_0', 'RandomUnderSampler_seed_1', 'RandomUnderSampler_seed_2', 'RandomUnderSampler_seed_3', 'RandomUnderSampler_seed_4', 'RandomUnderSampler_seed_5', 'RandomUnderSampler_seed_6', 'RandomUnderSampler_seed_7', 'RandomUnderSampler_seed_8', 'RandomUnderSampler_seed_9', 'TomekLinks']

METHOD: ENN

  Symmetric Excess AUC Results:
    Excess ROC (balanced|vanilla): 0.065718
    Excess ROC (vanilla|balanced): 0.002644
    Symmetric Excess ROC: 0.034181
    Coverage (informative vanilla leaves): 0.9924
    Coverage (informative balanced leaves): 0.9954

  Global Metrics:
    Vanilla OCT ROC-AUC: 0.566600
    Balanced OCT ROC-AUC: 0.623211
    Vanilla OCT PR-AUC: 0.086306
    Balanced OCT PR-AUC: 0.099154

  Bootstrap Confidence Intervals (95%)

TypeError: DataFrame.sort_values() got an unexpected keyword argument 'na_last'

In [6]:
print(f"\n{'='*80}")
print("SYMMETRIC EVALUATION COMPLETE")
print("="*80)

# Summary table of symmetric excess metrics
if len(symmetric_results) > 0:
    print("\n" + "="*80)
    print("SUMMARY: SYMMETRIC EXCESS AUC BY METHOD")
    print("="*80)
    
    summary_data = []
    for method_name, results in symmetric_results.items():
        if 'scores' in results:
            scores = results['scores']
            summary_data.append({
                'method': method_name,
                'excess_ROC_s|v': scores.get('excess_ROC_s|v', np.nan),
                'excess_ROC_v|s': scores.get('excess_ROC_v|s', np.nan),
                'sym_excess_ROC': scores.get('sym_excess_ROC', np.nan),
                'coverage_v': scores.get('coverage_informative_v', np.nan),
                'coverage_s': scores.get('coverage_informative_s', np.nan),
            })
    
    if summary_data:
        symmetric_summary_df = pd.DataFrame(summary_data)
        print("\n" + symmetric_summary_df.to_string(index=False))
        
        # Rank by symmetric excess ROC
        print("\n" + "="*80)
        print("RANKING BY SYMMETRIC EXCESS ROC (descending)")
        print("="*80)
        sorted_symmetric = symmetric_summary_df.sort_values('sym_excess_ROC', ascending=False, )
        print(sorted_symmetric.to_string(index=False))
        
        # Save symmetric evaluation results
        symmetric_path = "competing_methods_results/symmetric_evaluation_results.csv"
        symmetric_summary_df.to_csv(symmetric_path, index=False)
        print(f"\n✓ Symmetric evaluation results saved to: {symmetric_path}")


SYMMETRIC EVALUATION COMPLETE

SUMMARY: SYMMETRIC EXCESS AUC BY METHOD

                   method  excess_ROC_s|v  excess_ROC_v|s  sym_excess_ROC  coverage_v  coverage_s
                      ENN        0.065718        0.002644        0.034181    0.992448    0.995429
                      NCL        0.162518        0.002016        0.082267    0.992448    0.987281
               NearMiss-1        0.267578        0.017600        0.142589    0.992448    0.985294
               NearMiss-2       -0.089037        0.042301       -0.023368    0.992448    0.969595
               NearMiss-3       -0.254397        0.014376       -0.120010    0.992448    0.956479
       RandomUnderSampler        0.322761        0.012110        0.167436    0.992448    1.000000
               TomekLinks        0.059487        0.000705        0.030096    0.992448    0.989269
RandomUnderSampler_seed_0        0.299767        0.034661        0.167214    0.992448    0.580485
RandomUnderSampler_seed_1        0.288408    

## Vanilla model

In [ ]:
# ============================================================================
# TRAIN VANILLA OCT BASELINE (on full imbalanced training data)
# ============================================================================

print("="*80)
print("TRAINING VANILLA OCT BASELINE")
print("="*80)
print("Training on full imbalanced training data for comparison...")

try:
    # Train vanilla OCT on full imbalanced training data
    vanilla_model, vanilla_params, _, vanilla_preprocessor, vanilla_feature_names = finetune_oct(
        X_train=train_pd[feature_cols],
        y_train=train_pd[target_col],
        X_val=X_val,
        y_val=y_val,
        categorical_cols=CAT_COLUMNS,
        numeric_cols=TRUE_NUM_COLUMNS,
        depths=OCT_DEPTHS,
        minbuckets=OCT_MINBUCKETS,
        cps=OCT_CPS,
    )
    
    print(f"✓ Vanilla OCT training complete")
    print(f"  Best params: {vanilla_params}")
    
    # Evaluate and save predictions for vanilla model
    vanilla_results_dir = "vanilla_oct"
    vanilla_metrics = evaluate_binary_oct(
        vanilla_model, X_test, y_test, vanilla_preprocessor, vanilla_feature_names,
        results_dir=vanilla_results_dir, ratio=None
    )
    
    print(f"\n✓ Vanilla OCT predictions saved")
    print(f"  Predictions path: {vanilla_results_dir}/predictions/oct_predictions.csv")
    print(f"  AUC: {vanilla_metrics.get('auc', 'N/A'):.4f}" if isinstance(vanilla_metrics.get('auc'), (int, float)) else f"  AUC: {vanilla_metrics.get('auc', 'N/A')}")
    print(f"  PR-AUC: {vanilla_metrics.get('pr_auc', 'N/A'):.4f}" if isinstance(vanilla_metrics.get('pr_auc'), (int, float)) else f"  PR-AUC: {vanilla_metrics.get('pr_auc', 'N/A')}")
    print(f"  MCC: {vanilla_metrics.get('best_mcc', 'N/A'):.4f}" if isinstance(vanilla_metrics.get('best_mcc'), (int, float)) else f"  MCC: {vanilla_metrics.get('best_mcc', 'N/A')}")
    
except Exception as e:
    print(f"✗ ERROR training vanilla OCT: {e}")
    import traceback
    traceback.print_exc()